In [16]:
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from dataclasses import dataclass, field, asdict
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import utaeT
import utils
from weight_init import weight_init


In [17]:
@dataclass
class Config:
    # Model parameters
    input_dim = 5
    encoder_widths: list = field(default_factory=lambda: [16, 32, 32, 32])
    decoder_widths: list = field(default_factory=lambda: [16, 16, 32, 32])
    out_conv: int = 1

    str_conv_k: int = 4
    str_conv_s: int = 2
    str_conv_p: int = 1

    agg_mode: str = "att_group"
    encoder_norm: str = "group"

    n_head: int = 4
    d_model: int = 32
    d_k: int = 4

    # Dataset / paths
    path_sentinel_data: Path = field(default_factory=lambda: Path.cwd().parents[3] / "ideas-dslab-group1-shared" / "preprocessed data" / "satellite_patches_vie.npy")
    path_tree_data: Path = field(default_factory=lambda: Path.cwd().parents[3] / "ideas-dslab-group1-shared" / "preprocessed data" / "tree_patches_vie.npy")
    path_date_data: Path = field(default_factory=lambda: Path.cwd().parents[3] / "ideas-dslab-group1-shared" / "preprocessed data" / "date_patches_vie.n)
    res_dir: Path = field(default_factory=lambda: Path.cwd() / "results")

    num_workers: int = 4
    rdm_seed: int = 1
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    def torch_device(self):
        return torch.device(self.device)

    cache: bool = False

    # Training
    epochs: int = 100
    batch_size: int = 16
    lr: float = 0.001

    num_classes: int = 20
    ignore_index: int = -1
    pad_value: float = 0.0
    padding_mode: str = "reflect"

    val_every: int = 1
    val_after: int = 0
    display_step: int = 1

    fold: int = None

In [18]:
class TreeDataset(Dataset):
    # generates the combined dataset with inputs, dates and labels

    def __init__(self, patches, dates, masks):
        """
        patches: (N, T, C, H, W)
        dates:   (N, T)
        masks:   (N, H, W)
        """
        self.patches = patches
        self.dates = dates
        self.masks = masks


    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        x = self.patches[idx]
        y = self.masks[idx]
        date = self.dates[idx]

        # convert to torch tensors
        x = torch.tensor(x, dtype=torch.float32)
        date = torch.tensor(date, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32).unsqueeze(0)

        return x, date, y

In [19]:
def recursive_todevice(x, device):
    # put data to right device
    if isinstance(x, torch.Tensor):
        return x.to(device)
    elif isinstance(x, dict):
        return {k: recursive_todevice(v, device) for k, v in x.items()}
    else:
        return [recursive_todevice(c, device) for c in x]


In [20]:
class AverageMeter:
    # get the average of metrics

    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0
        self.count = 0

    def add(self, val):
        self.sum += val
        self.count += 1

    def value(self):
        return self.sum / max(self.count, 1)

In [21]:
def iterate(model, data_loader, criterion, config, optimizer=None, mode="train"):
    # iterates over each batch in the data_loader and trains/tests model

    loss_meter = AverageMeter()
    mae_meter = AverageMeter()
    rmse_meter = AverageMeter()
    all_preds = []
    all_targets = []

    t_start = time.time()
    for i, batch in enumerate(data_loader):
        batch = recursive_todevice(batch, config.device)
        x, dates, y = batch
        y = y.float()

        if mode != "train":
            with torch.no_grad():
                out = model(x, batch_positions=dates)
        else:
            optimizer.zero_grad()
            out = model(x, batch_positions=dates)

        loss = criterion(out, y)

        with torch.no_grad():
            mae = torch.mean(torch.abs(out - y))
            rmse = torch.sqrt(torch.mean((out - y) ** 2))

            pred = out.detach().cpu().numpy()
            true = y.detach().cpu().numpy()
            all_preds.append(pred)
            all_targets.append(true)

        if mode == "train":
            loss.backward()
            optimizer.step()

        mae_meter.add(mae.item())
        rmse_meter.add(rmse.item())
        loss_meter.add(loss.item())

        if (i + 1) % config.display_step == 0:
            print(f"Step [{i + 1}/{len(data_loader)}], Loss: {loss_meter.value():.4f}, RMSE: {rmse_meter.value():.4f}")

    t_end = time.time()
    total_time = t_end - t_start
    print(f"Epoch time: {total_time:.1f}")
    metrics = {
        f"{mode}_loss": loss_meter.value(),
        f"{mode}_mae": mae_meter.value(),
        f"{mode}_rmse": rmse_meter.value()
        }

    return metrics, all_preds, all_targets

In [22]:
def prepare_output(config):
    os.makedirs(config.res_dir, exist_ok=True)
    for fold in range(1, 6):
        os.makedirs(os.path.join(config.res_dir, "Fold_{}".format(fold)), exist_ok=True)

In [23]:
def checkpoint(fold, log, config):
    # create checkpoints to save best model for validation
    with open(
        os.path.join(config.res_dir, "Fold_{}".format(fold), "trainlog.json"), "w"
    ) as outfile:
        json.dump(log, outfile, indent=4)

In [24]:
def config_to_json(config):
    # helper function to get config data to json file for reproducibility

    d = asdict(config)

    for k, v in d.items():
        if isinstance(v, Path):
            d[k] = str(v)

    return d

In [25]:
def save_results(fold, metrics, preds, targets, config):
    # saving the results

    out_dir = os.path.join(config.res_dir, f"Fold_{fold}")
    os.makedirs(out_dir, exist_ok=True)

    # 1. Save metrics (MAE, RMSE, R2)
    with open(os.path.join(out_dir, "test_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=4)

    # 2. Save predictions + ground truth 
    np.save(
        os.path.join(out_dir, "predictions.npy"),
        np.array(preds, dtype=np.float32)
    )

    np.save(
        os.path.join(out_dir, "targets.npy"),
        np.array(targets, dtype=np.float32)
    )

    # 3. save config for reproducibility
    with open(os.path.join(out_dir, "config.json"), "w") as f:
        json.dump(config_to_json(config), f, indent=4)

In [26]:
def overall_performance(all_preds, all_targets):
    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_targets, axis=0)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    metrics = {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    }

    print("\n=== Overall Regression Performance ===")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")

    return metrics

In [27]:
def split_into_5_blocks(N):
    # split all indices of data set into 5 blocks for train/val/test split

    indices = np.arange(N)

    # split into 5 (nearly) equal parts
    blocks = np.array_split(indices, 5)

    return blocks


In [28]:
def make_fold_sequence(blocks):
    # create the train/val/test split fold sequence

    fold_sequence = []

    for i in range(len(blocks)):

        train = np.concatenate([
            blocks[(i + 0) % 5],
            blocks[(i + 1) % 5],
            blocks[(i + 2) % 5],
        ])

        val = blocks[(i + 3) % 5]
        test = blocks[(i + 4) % 5]

        fold_sequence.append([train, val, test])

    return fold_sequence

In [29]:
def main(config):
    # set random seed and device
    np.random.seed(config.rdm_seed)
    torch.manual_seed(config.rdm_seed)
    device = torch.device(config.device)

    # import data 
    sentinel_data = np.load(config.path_sentinel_data)
    tree_data = np.load(config.path_tree_data)
    date_data = np.load(config.path_date_data)

    # convert nan values to 0
    sentinel_data = np.nan_to_num(sentinel_data, nan=0.0)

    # check for nans
    print("x NaNs:", np.isnan(sentinel_data).any())
    print("y NaNs:", np.isnan(tree_data).any())
    print("dates NaNs:", np.isnan(date_data).any())

    # convert to pytorch Dataset
    dataset = TreeDataset(sentinel_data, date_data, tree_data)

    # get indices of train/validation/test data for 5 folds
    blocks = split_into_5_blocks(len(dataset))
    fold_sequence = make_fold_sequence(blocks)

    # create lists for test evaluation
    all_preds_folds = []
    all_targets_folds = []
    val_preds_folds = []
    val_targets_folds = []

    # prepare output folders
    prepare_output(config)

    for fold, (train_idx, val_idx, test_idx) in enumerate(fold_sequence):
        # get the split data
        dt_train = Subset(dataset, train_idx)
        dt_val   = Subset(dataset, val_idx)
        dt_test  = Subset(dataset, test_idx)

        # create dataloaders using padding
        collate_fn = lambda x: utils.pad_collate(x, pad_value=config.pad_value)
        train_loader = DataLoader(
            dt_train,
            batch_size=config.batch_size,
            shuffle=True,
            drop_last=True,
            collate_fn=collate_fn,
        )
        val_loader = DataLoader(
            dt_val,
            batch_size=config.batch_size,
            shuffle=False,
            drop_last=False,
            collate_fn=collate_fn,
        )
        test_loader = DataLoader(
            dt_test,
            batch_size=config.batch_size,
            shuffle=False,
            drop_last=False,
            collate_fn=collate_fn,
        )

        print(f"Train {len(dt_train)}, Val {len(dt_val)}, Test {len(dt_test)}")

        # Model definition
        model = utaeT.UTAE(config)

        # get number of parameters
        config.N_params = utils.get_ntrainparams(model)

        # safe config for future reproducibility
        with open(os.path.join(config.res_dir, "conf.json"), "w") as file:
            json.dump(config_to_json(config), file, indent=4)

        # print info about the model
        print(model)
        print("TOTAL TRAINABLE PARAMETERS :", config.N_params)
        print("Trainable layers:")
        for name, p in model.named_parameters():
            if p.requires_grad:
                print(name)

        # get model on right device and initialize weights
        model = model.to(device)
        model.apply(weight_init)

        # Optimizer and Loss
        optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
        criterion = nn.SmoothL1Loss()

        # Training loop
        trainlog = {}
        best_loss = float("inf")
        for epoch in range(1, config.epochs + 1):
            print(f"EPOCH {epoch}/{config.epochs}")

            model.train()
            train_metrics, _, _ = iterate(
                model,
                data_loader=train_loader,
                criterion=criterion,
                config=config,
                optimizer=optimizer,
                mode="train"
            )
            if epoch % config.val_every == 0 and epoch > config.val_after:
                print("Validation . . . ")
                model.eval()
                val_metrics, val_preds, val_targets = iterate(
                    model,
                    data_loader=val_loader,
                    criterion=criterion,
                    config=config,
                    optimizer=optimizer,
                    mode="val"
                )
                val_preds_folds.append(val_preds)
                val_targets_folds.append(val_targets)

                print(f"Loss: {val_metrics['val_loss']:.4f}")

                trainlog[epoch] = {**train_metrics, **val_metrics}
                checkpoint(fold + 1, trainlog, config)
                if val_metrics["val_loss"] < best_loss:
                    best_loss = val_metrics["val_loss"]
                    torch.save(
                        {
                            "epoch": epoch,
                            "state_dict": model.state_dict(),
                            "optimizer": optimizer.state_dict(),
                        },
                        os.path.join(config.res_dir, f"Fold_{fold + 1}","model.pth.tar"),
                    )
            else:
                trainlog[epoch] = {**train_metrics}
                checkpoint(fold + 1, trainlog, config)

        print("Testing best epoch . . .")
        model.load_state_dict(
            torch.load(
                os.path.join(
                    config.res_dir, f"Fold_{fold + 1}", "model.pth.tar"
                )
            )["state_dict"]
        )
        model.eval()

        test_metrics, preds, targets = iterate(
            model,
            data_loader=test_loader,
            criterion=criterion,
            config=config,
            optimizer=optimizer,
            mode="test",
        )
        all_preds_folds.append(preds)
        all_targets_folds.append(targets)
        print(f"Loss {test_metrics['test_loss']:.4f}")
        save_results(fold + 1,test_metrics,preds,targets,config)

    if config.fold is None:
        overall_performance(all_preds_folds, all_targets_folds)

In [30]:
if __name__ == "__main__":
    config = Config()
    main(config)

x NaNs: False
y NaNs: False
dates NaNs: False
Train 356, Val 118, Test 118
UTAE(
  (in_conv): ConvBlock(
    (conv): ConvLayer(
      (conv): Sequential(
        (0): Conv2d(5, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (1): GroupNorm(4, 16, eps=1e-05, affine=True)
        (2): ReLU()
        (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
        (4): GroupNorm(4, 16, eps=1e-05, affine=True)
        (5): ReLU()
      )
    )
  )
  (down_blocks): ModuleList(
    (0): DownConvBlock(
      (down): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(16, 16, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), padding_mode=reflect)
          (1): GroupNorm(4, 16, eps=1e-05, affine=True)
          (2): ReLU()
        )
      )
      (conv1): ConvLayer(
        (conv): Sequential(
          (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), padding_mode=reflect)
          (1): Gr

KeyboardInterrupt: 